## Etapa 1: ETL e Limpeza de Dados

In [ ]:
# ===================================================================
# PROJETO: Spotify Weekly Top 200 - ETL e Limpeza de Dados
# OBJETIVO: Limpeza e preparação dos dados para análise
# ===================================================================

print("""
╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║   🎵 SPOTIFY WEEKLY TOP 200 - ETL E LIMPEZA DE DADOS 🎵          ║
║                                                                  ║
║   Processamento de dados musicais para análise de tendências     ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║   🎵 SPOTIFY WEEKLY TOP 200 - ETL E LIMPEZA DE DADOS 🎵          ║
║                                                                  ║
║   Processamento de dados musicais para análise de tendências     ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝



In [ ]:
# ===================================================================
# IMPORTAR BIBLIOTECAS
# ===================================================================

import pandas as pd
import numpy as np
import warnings
import os
import sys

# Configurações
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Bibliotecas importadas com sucesso!")
print(f"   Pandas versão: {pd.__version__}")
print(f"   Numpy versão: {np.__version__}")
print(f"   Python versão: {sys.version.split()[0]}")

✅ Bibliotecas importadas com sucesso!
   Pandas versão: 2.2.2
   Numpy versão: 2.0.2
   Python versão: 3.12.13


In [ ]:
# ===================================================================
# MONTAR GOOGLE DRIVE
# ===================================================================
# Explicação: O arquivo está no Google Drive, precisamos montá-lo para acessar os dados.

from google.colab import drive

print("🔗 Montando Google Drive...")
drive.mount('/content/drive')

print("\n✅ Drive montado com sucesso!")
print("   Arquivos disponíveis em: content/drive/MyDrive")

🔗 Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Drive montado com sucesso!
   Arquivos disponíveis em: content/drive/MyDrive


In [ ]:
# ===================================================================
# VERIFICAR ARQUIVO DE DADOS
# ===================================================================

# Caminho do arquivo
ARQUIVO = '/content/drive/MyDrive/final.csv'

print("📂 Verificando arquivo...")
print("-" * 50)

# Verificar se o arquivo existe
if os.path.exists(ARQUIVO):
    tamanho_bytes = os.path.getsize(ARQUIVO)
    tamanho_mb = tamanho_bytes / (1024**2)
    tamanho_gb = tamanho_bytes / (1024**3)

    print(f"✅ Arquivo encontrado!")
    print(f"   📁 Caminho: {ARQUIVO}")
    print(f"   💾 Tamanho: {tamanho_mb:.2f} MB ({tamanho_gb:.2f} GB)")

    # Verificar primeiras linhas
    print("\n🔍 Visualizando primeiras linhas do arquivo:")
    print("-" * 50)
    amostra = pd.read_csv(ARQUIVO, nrows=3)
    print(amostra)

else:
    print(f"❌ ERRO: Arquivo não encontrado em: {ARQUIVO}")
    print("\n📁 Arquivos disponíveis na pasta MyDrive:")
    !ls -la /content/drive/MyDrive/*.csv 2>/dev/null || echo "Nenhum arquivo CSV encontrado"
    raise FileNotFoundError("Arquivo não encontrado! Verifique o caminho.")

📂 Verificando arquivo...
--------------------------------------------------
✅ Arquivo encontrado!
   📁 Caminho: /content/drive/MyDrive/final.csv
   💾 Tamanho: 796.63 MB (0.78 GB)

🔍 Visualizando primeiras linhas do arquivo:
--------------------------------------------------
   Unnamed: 0                                   uri  rank  artist_names  \
0           0  spotify:track:2gpQi3hbcUAcEG8m2dlgfB     1  Paulo Londra   
1           1  spotify:track:2x8oBuYaObjqHqgGuIUZ0b     2           WOS   
2           2  spotify:track:2SJZdZ5DLtlRosJ2xHJJJa     3  Paulo Londra   

   artists_num artist_individual                              artist_id  \
0         1.00      Paulo Londra  spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z   
1         1.00               WOS  spotify:artist:5YCc6xS5Gpj3EkaYGdjyNK   
2         1.00      Paulo Londra  spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z   

        artist_genre                                         artist_img  \
0  argentine hip hop  https://i.scdn.co/image/ab

In [ ]:
# ===================================================================
# Carregar dados: TODOS OS PAÍSES DA AMÉRICA
# Carregamento inteligente com chunks para não travar
# ===================================================================

print("🌎 CARREGANDO TODOS OS DADOS DAS AMÉRICAS...")
print("=" * 60)

# Lista de países da América
PAISES_AMERICA = [
    'United States', 'USA', 'Canada', 'Mexico',
    'Brazil', 'Brasil', 'Argentina', 'Chile', 'Peru', 'Colombia',
    'Venezuela', 'Ecuador', 'Bolivia', 'Paraguay', 'Uruguay',
    'Costa Rica', 'Panama', 'Guatemala', 'Dominican Republic', 'Cuba'
]

COLUNAS_SELECIONADAS = [
    'rank', 'artist_names', 'track_name', 'release_date',
    'streams', 'week', 'country', 'region', 'language',
    'danceability', 'valence', 'energy'
]

# Carregar em chunks para evitar travamentos
chunk_size = 100000
chunks_america = []

print("🔄 Processando arquivo em chunks...")

for i, chunk in enumerate(pd.read_csv(
    ARQUIVO,
    usecols=COLUNAS_SELECIONADAS,
    chunksize=chunk_size,
    low_memory=False
)):
    # Filtrar apenas países da América
    chunk_filtrado = chunk[chunk['country'].isin(PAISES_AMERICA)]

    if len(chunk_filtrado) > 0:
        chunks_america.append(chunk_filtrado)
        total_registros = sum(len(c) for c in chunks_america)
        print(f"   Chunk {i+1:3}: +{len(chunk_filtrado):6,} registros (total: {total_registros:8,})")

    # Limpar memória a cada 10 chunks
    if (i + 1) % 10 == 0:
        import gc
        gc.collect()

# Concatenar todos os chunks
if chunks_america:
    df = pd.concat(chunks_america, ignore_index=True)
    print(f"\n✅ Carregamento concluído!")
    print(f"   📊 Total de registros das Américas: {len(df):,}")
    print(f"   🌍 Países encontrados: {df['country'].nunique()}")
    print(f"   💾 Memória utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
else:
    print("❌ Nenhum dado das Américas encontrado!")
    df = pd.DataFrame()

# Limpar memória
del chunks_america
import gc
gc.collect()

# Mostrar países encontrados
print(f"\n🌍 PAÍSES ENCONTRADOS:")
paises_encontrados = sorted(df['country'].unique())
for pais in paises_encontrados:
    qtd = len(df[df['country'] == pais])
    print(f"   • {pais:20} : {qtd:6,} registros")

# Continuar com o resto da análise...
# Agora você pode usar o DataFrame 'df' para todas as análises

🌎 CARREGANDO TODOS OS DADOS DAS AMÉRICAS...
🔄 Processando arquivo em chunks...
   Chunk   1: +29,476 registros (total:   29,476)
   Chunk   2: +71,629 registros (total:  101,105)
   Chunk   3: +98,536 registros (total:  199,641)
   Chunk   4: +30,771 registros (total:  230,412)
   Chunk   5: +28,235 registros (total:  258,647)
   Chunk   7: + 2,201 registros (total:  260,848)
   Chunk   8: +25,896 registros (total:  286,744)
   Chunk  11: +25,182 registros (total:  311,926)
   Chunk  13: +77,320 registros (total:  389,246)
   Chunk  14: +12,998 registros (total:  402,244)
   Chunk  18: +59,323 registros (total:  461,567)

✅ Carregamento concluído!
   📊 Total de registros das Américas: 461,567
   🌍 Países encontrados: 17
   💾 Memória utilizada: 317.30 MB

🌍 PAÍSES ENCONTRADOS:
   • Argentina            : 29,476 registros
   • Bolivia              : 29,054 registros
   • Brazil               : 29,537 registros
   • Canada               : 22,856 registros
   • Chile                : 32,31

In [ ]:
# ===================================================================
# ANÁLISE EXPLORATÓRIA INICIAL
# ===================================================================
# Objetivo: Entender a estrutura e qualidade dos dados antes da limpeza

print("🔍 ANÁLISE EXPLORATÓRIA INICIAL")
print("=" * 50)

# 5.1 Estrutura dos dados
print("\n1️⃣ ESTRUTURA DOS DADOS:")
print("-" * 30)
print(f"Total de registros: {len(df):,}")
print(f"Total de colunas: {len(df.columns)}")
print(f"\nColunas disponíveis: {list(df.columns)}")

# 5.2 Tipos de dados
print("\n2️⃣ TIPOS DE DADOS:")
print("-" * 30)
print(df.dtypes)

# 5.3 Valores nulos
print("\n3️⃣ VALORES NULOS:")
print("-" * 30)
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]

if len(nulos) > 0:
    print("⚠️ Valores nulos encontrados:")
    for col, qtd in nulos.items():
        print(f"   • {col:15} : {qtd:6,} ({qtd/len(df)*100:.1f}%)")
else:
    print("✅ Nenhum valor nulo encontrado!")

# 5.4 Primeiras linhas
print("\n4️⃣ PRIMEIRAS LINHAS:")
print("-" * 30)
print(df.head())

# 5.5 Estatísticas descritivas
print("\n5️⃣ ESTATÍSTICAS DESCRITIVAS (Colunas Numéricas):")
print("-" * 30)
print(df.describe())

# 5.6 Valores únicos por coluna categórica
print("\n6️⃣ VALORES ÚNICOS (Colunas Categóricas):")
print("-" * 30)
categoricas = ['country', 'region', 'language']
for col in categoricas:
    if col in df.columns:
        print(f"   • {col:12} : {df[col].nunique():3} valores únicos")
        if df[col].nunique() <= 10:
            print(f"     Valores: {df[col].unique().tolist()}")

🔍 ANÁLISE EXPLORATÓRIA INICIAL

1️⃣ ESTRUTURA DOS DADOS:
------------------------------
Total de registros: 461,567
Total de colunas: 12

Colunas disponíveis: ['rank', 'artist_names', 'track_name', 'release_date', 'streams', 'week', 'danceability', 'energy', 'valence', 'country', 'region', 'language']

2️⃣ TIPOS DE DADOS:
------------------------------
rank            object
artist_names    object
track_name      object
release_date    object
streams         object
week            object
danceability    object
energy          object
valence         object
country         object
region          object
language        object
dtype: object

3️⃣ VALORES NULOS:
------------------------------
⚠️ Valores nulos encontrados:
   • danceability    :    172 (0.0%)
   • energy          :    172 (0.0%)
   • valence         :    172 (0.0%)

4️⃣ PRIMEIRAS LINHAS:
------------------------------
  rank  artist_names             track_name release_date  streams        week  \
0    1  Paulo Londra        

In [ ]:
# ===================================================================
# CONVERSÃO DE TIPOS DE DADOS
# ===================================================================
# Explicação: Muitas colunas estão como 'object' (texto) quando
# deveriam ser números. Vamos converter para os tipos corretos.

print("🔄 CONVERSÃO DE TIPOS DE DADOS")
print("=" * 50)

# Registrar memória antes
memoria_antes = df.memory_usage(deep=True).sum() / 1024**2

print(f"💾 Memória antes da conversão: {memoria_antes:.2f} MB\n")

# 6.1 Converter colunas numéricas
print("1️⃣ Convertendo colunas numéricas:")
print("-" * 30)

colunas_numericas = ['rank', 'streams', 'danceability', 'energy', 'valence']

for col in colunas_numericas:
    if col in df.columns:
        # Converter para numérico, forçando erros para NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"   ✅ {col}: convertido para numérico")

# 6.2 Converter datas
print("\n2️⃣ Convertendo datas:")
print("-" * 30)

if 'week' in df.columns:
    df['week'] = pd.to_datetime(df['week'], errors='coerce')
    print(f"   ✅ week: convertido para datetime")

if 'release_date' in df.columns:
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    print(f"   ✅ release_date: convertido para datetime")

# 6.3 Otimizar colunas categóricas (economiza memória)
print("\n3️⃣ Otimizando colunas categóricas:")
print("-" * 30)

colunas_categoricas = ['country', 'region', 'language']

for col in colunas_categoricas:
    if col in df.columns:
        # Converter para category (mais eficiente)
        df[col] = df[col].astype('category')
        print(f"   ✅ {col}: convertido para category")

# Verificar economia de memória
memoria_depois = df.memory_usage(deep=True).sum() / 1024**2
economia = ((memoria_antes - memoria_depois) / memoria_antes) * 100

print(f"\n💾 Memória depois da conversão: {memoria_depois:.2f} MB")
print(f"💰 Economia de memória: {economia:.1f}%")

print("\n✅ Tipos convertidos com sucesso!")

🔄 CONVERSÃO DE TIPOS DE DADOS
💾 Memória antes da conversão: 317.30 MB

1️⃣ Convertendo colunas numéricas:
------------------------------
   ✅ rank: convertido para numérico
   ✅ streams: convertido para numérico
   ✅ danceability: convertido para numérico
   ✅ energy: convertido para numérico
   ✅ valence: convertido para numérico

2️⃣ Convertendo datas:
------------------------------
   ✅ week: convertido para datetime
   ✅ release_date: convertido para datetime

3️⃣ Otimizando colunas categóricas:
------------------------------
   ✅ country: convertido para category
   ✅ region: convertido para category
   ✅ language: convertido para category

💾 Memória depois da conversão: 91.83 MB
💰 Economia de memória: 71.1%

✅ Tipos convertidos com sucesso!


In [ ]:
# ===================================================================
# TRATAMENTO DE VALORES NULOS
# ===================================================================
# Explicação: Decidimos como tratar cada tipo de valor nulo:
# - Datas essenciais (week): remover linhas
# - Dados numéricos: preencher com mediana
# - Datas não essenciais: preencher com moda (data mais comum)

print("🔧 TRATAMENTO DE VALORES NULOS")
print("=" * 50)

# Contar nulos antes
nulos_antes = df.isnull().sum().sum()
print(f"📊 Total de células nulas antes: {nulos_antes:,}")

# 7.1 Remover linhas com 'week' nula (essencial para análise temporal)
print("\n1️⃣ Removendo linhas com 'week' nula:")
print("-" * 30)

antes = len(df)
df = df.dropna(subset=['week'])
removidos = antes - len(df)
print(f"   ❌ Linhas removidas: {removidos:,}")
print(f"   ✅ Linhas restantes: {len(df):,}")

# 7.2 Preencher nulos numéricos com a MEDIANA
print("\n2️⃣ Preenchendo nulos numéricos com MEDIANA:")
print("-" * 30)
print("   (Mediana é mais robusta contra outliers)")

for col in colunas_numericas:
    if col in df.columns and df[col].isnull().sum() > 0:
        mediana = df[col].median()
        nulos = df[col].isnull().sum()
        df[col].fillna(mediana, inplace=True)
        print(f"   • {col:12} : {nulos:4} nulos → preenchidos com mediana={mediana:.3f}")

# 7.3 Preencher 'release_date' com a MODA (data mais comum)
print("\n3️⃣ Preenchendo 'release_date' com MODA:")
print("-" * 30)
print("   (Data mais comum representa padrão da indústria)")

if 'release_date' in df.columns and df['release_date'].isnull().sum() > 0:
    data_comum = df['release_date'].mode()[0]
    nulos = df['release_date'].isnull().sum()
    df['release_date'].fillna(data_comum, inplace=True)
    print(f"   • release_date: {nulos} nulos → preenchidos com {data_comum.date()}")

# Verificar resultado
nulos_depois = df.isnull().sum().sum()
print(f"\n✅ Total de células nulas depois: {nulos_depois:,}")

if nulos_depois == 0:
    print("🎉 PARABÉNS! Não há mais valores nulos no dataset!")
else:
    print(f"⚠️ Ainda existem {nulos_depois} células nulas.")
    print(df.isnull().sum()[df.isnull().sum() > 0])

🔧 TRATAMENTO DE VALORES NULOS
📊 Total de células nulas antes: 4,151

1️⃣ Removendo linhas com 'week' nula:
------------------------------
   ❌ Linhas removidas: 0
   ✅ Linhas restantes: 461,567

2️⃣ Preenchendo nulos numéricos com MEDIANA:
------------------------------
   (Mediana é mais robusta contra outliers)
   • danceability :  172 nulos → preenchidos com mediana=0.744
   • energy       :  172 nulos → preenchidos com mediana=0.691
   • valence      :  172 nulos → preenchidos com mediana=0.634

3️⃣ Preenchendo 'release_date' com MODA:
------------------------------
   (Data mais comum representa padrão da indústria)
   • release_date: 3635 nulos → preenchidos com 2020-02-29

✅ Total de células nulas depois: 0
🎉 PARABÉNS! Não há mais valores nulos no dataset!


In [ ]:
# ===================================================================
# REMOVER DUPLICATAS
# ===================================================================
# Explicação: Uma música não deveria aparecer duas vezes
# no mesmo país e mesma semana.

print("🔄 REMOVENDO DUPLICATAS")
print("=" * 50)

# 8.1 Duplicatas exatas (todas as colunas iguais)
print("1️⃣ Duplicatas exatas:")
print("-" * 30)

duplicatas_exatas = df.duplicated().sum()
print(f"   Duplicatas exatas: {duplicatas_exatas:,}")

# 8.2 Duplicatas semânticas (mesma música, país e semana)
print("\n2️⃣ Duplicatas semânticas:")
print("-" * 30)
print("   (mesma música, mesmo artista, mesmo país, mesma semana)")

colunas_chave = ['track_name', 'artist_names', 'country', 'week']
# Verificar se todas as colunas existem
colunas_chave_existentes = [col for col in colunas_chave if col in df.columns]

if len(colunas_chave_existentes) == len(colunas_chave):
    duplicatas_semanticas = df.duplicated(subset=colunas_chave).sum()
    print(f"   Duplicatas semânticas: {duplicatas_semanticas:,}")

    # Remover duplicatas
    antes = len(df)
    df = df.drop_duplicates(subset=colunas_chave, keep='first')
    removidas = antes - len(df)
    print(f"\n   ✅ Removidas: {removidas:,} linhas")
    print(f"   ✅ Linhas restantes: {len(df):,}")
else:
    print(f"   ⚠️ Colunas faltando: {set(colunas_chave) - set(colunas_chave_existentes)}")
    print("   Pulando remoção de duplicatas semânticas")

print("\n✅ Duplicatas removidas com sucesso!")

🔄 REMOVENDO DUPLICATAS
1️⃣ Duplicatas exatas:
------------------------------
   Duplicatas exatas: 214,993

2️⃣ Duplicatas semânticas:
------------------------------
   (mesma música, mesmo artista, mesmo país, mesma semana)
   Duplicatas semânticas: 214,996

   ✅ Removidas: 214,996 linhas
   ✅ Linhas restantes: 246,571

✅ Duplicatas removidas com sucesso!


In [ ]:
# ===================================================================
# TRATAMENTO DE OUTLIERS
# ===================================================================
# Explicação: Outliers são valores extremos que podem distorcer a análise.
# Vamos usar uma abordagem conservadora: remover apenas os 0.5% mais extremos.

print("📊 TRATAMENTO DE OUTLIERS")
print("=" * 50)

# 9.1 Analisar streams (pode ter hits legítimos)
print("1️⃣ Análise de OUTLIERS em STREAMS:")
print("-" * 30)

if 'streams' in df.columns:
    # Estatísticas
    q1 = df['streams'].quantile(0.25)
    q3 = df['streams'].quantile(0.75)
    iqr = q3 - q1

    print(f"   Q1 (25%): {q1:,.0f}")
    print(f"   Q3 (75%): {q3:,.0f}")
    print(f"   IQR: {iqr:,.0f}")
    print(f"   Média: {df['streams'].mean():,.0f}")
    print(f"   Mediana: {df['streams'].median():,.0f}")

    # Método conservador: remover apenas percentil 99.5%
    percentil_995 = df['streams'].quantile(0.995)
    outliers = df[df['streams'] > percentil_995]

    print(f"\n   Percentil 99.5%: {percentil_995:,.0f}")
    print(f"   Outliers detectados: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")

    # Remover outliers
    antes = len(df)
    df = df[df['streams'] <= percentil_995]
    removidos = antes - len(df)
    print(f"\n   ✂️ Removidas: {removidos:,} linhas com streams extremamente altos")
    print(f"   ✅ Linhas restantes: {len(df):,}")

# 9.2 Corrigir valores inválidos nas características musicais
print("\n2️⃣ Corrigindo características musicais (intervalo 0-1):")
print("-" * 30)

for col in ['danceability', 'energy', 'valence']:
    if col in df.columns:
        invalidos_abaixo = (df[col] < 0).sum()
        invalidos_acima = (df[col] > 1).sum()

        if invalidos_abaixo > 0 or invalidos_acima > 0:
            print(f"   • {col}: {invalidos_abaixo} abaixo de 0, {invalidos_acima} acima de 1")
            df[col] = df[col].clip(0, 1)
            print(f"     ✅ Corrigido para intervalo [0,1]")
        else:
            print(f"   • {col}: ✅ todos os valores válidos")

# 9.3 Verificar consistência do rank
print("\n3️⃣ Verificando consistência do RANK:")
print("-" * 30)

if 'rank' in df.columns:
    ranks_invalidos = ((df['rank'] < 1) | (df['rank'] > 200)).sum()
    if ranks_invalidos > 0:
        print(f"   ⚠️ Ranks inválidos: {ranks_invalidos}")
        antes = len(df)
        df = df[(df['rank'] >= 1) & (df['rank'] <= 200)]
        print(f"   ✅ Removidas {antes - len(df)} linhas com rank inválido")
    else:
        print(f"   ✅ Todos os ranks entre 1 e 200")
        print(f"   Rank mínimo: {df['rank'].min()}")
        print(f"   Rank máximo: {df['rank'].max()}")

print("\n✅ Outliers tratados com sucesso!")

📊 TRATAMENTO DE OUTLIERS
1️⃣ Análise de OUTLIERS em STREAMS:
------------------------------
   Q1 (25%): 34,594
   Q3 (75%): 426,326
   IQR: 391,732
   Média: 470,710
   Mediana: 107,598

   Percentil 99.5%: 5,547,964
   Outliers detectados: 1,233 (0.50%)

   ✂️ Removidas: 1,233 linhas com streams extremamente altos
   ✅ Linhas restantes: 245,338

2️⃣ Corrigindo características musicais (intervalo 0-1):
------------------------------
   • danceability: ✅ todos os valores válidos
   • energy: ✅ todos os valores válidos
   • valence: ✅ todos os valores válidos

3️⃣ Verificando consistência do RANK:
------------------------------
   ✅ Todos os ranks entre 1 e 200
   Rank mínimo: 1
   Rank máximo: 200

✅ Outliers tratados com sucesso!


In [ ]:
# ===================================================================
# FEATURE ENGINEERING - ADAPTADO PARA ANÁLISE DAS AMÉRICAS
# ===================================================================
# Explicação: Criar novas colunas que facilitam a análise e visualização
# Contexto: Foco em características relevantes para o mercado latino-americano

print("🔨 FEATURE ENGINEERING - ANÁLISE DAS AMÉRICAS")
print("=" * 60)

# Verificar se df existe e tem dados
if df is None or len(df) == 0:
    print("❌ DataFrame vazio! Execute o carregamento dos dados primeiro.")
    raise ValueError("DataFrame não carregado")

print(f"📊 Shape inicial: {df.shape}")
print(f"🌍 Países presentes: {df['country'].nunique()}")

# ===================================================================
# 1. STREAMS EM MILHÕES (mais legível)
# ===================================================================
print("\n1️⃣ Streams em milhões:")
print("-" * 40)

if 'streams' in df.columns:
    df['streams_millions'] = (df['streams'] / 1_000_000).round(2)
    print(f"   ✅ streams_millions criada")
    print(f"   📊 Total de streams nas Américas: {df['streams_millions'].sum():.1f}M")
    print(f"   📊 Média por música: {df['streams_millions'].mean():.2f}M")
else:
    print("   ❌ Coluna 'streams' não encontrada!")

# ===================================================================
# 2. COMPONENTES DA DATA (para análise temporal)
# ===================================================================
print("\n2️⃣ Componentes da data:")
print("-" * 40)

if 'week' in df.columns:
    # Converter para datetime se não for
    if not pd.api.types.is_datetime64_any_dtype(df['week']):
        df['week'] = pd.to_datetime(df['week'], errors='coerce')

    df['year'] = df['week'].dt.year
    df['month'] = df['week'].dt.month
    df['month_name'] = df['week'].dt.strftime('%B')  # Nome do mês
    df['day_of_week'] = df['week'].dt.dayofweek  # 0=Segunda, 6=Domingo
    df['day_name'] = df['week'].dt.day_name()  # Nome do dia
    df['week_number'] = df['week'].dt.isocalendar().week

    print(f"   ✅ year: {df['year'].min()} - {df['year'].max()}")
    print(f"   ✅ month: criada")
    print(f"   ✅ month_name: {df['month_name'].iloc[0] if len(df) > 0 else 'N/A'}")
    print(f"   ✅ day_of_week: 0=Segunda, 6=Domingo")
    print(f"   ✅ day_name: criada")
    print(f"   ✅ week_number: criada")
else:
    print("   ❌ Coluna 'week' não encontrada!")

# ===================================================================
# 3. SCORE DE SUCESSO (combina rank e streams)
# ===================================================================
print("\n3️⃣ Score de sucesso (Rank ajustado):")
print("-" * 40)
print("   Fórmula: (201 - rank) * log(streams_millions)")
print("   Quanto MAIOR o score, MELHOR a música")

if 'rank' in df.columns and 'streams_millions' in df.columns:
    # Evitar log de zero
    df['success_score'] = (201 - df['rank']) * np.log1p(df['streams_millions'])
    print(f"   ✅ success_score criada")
    print(f"   📊 Média: {df['success_score'].mean():.2f}")
    print(f"   📊 Mediana: {df['success_score'].median():.2f}")
    print(f"   📊 Mínimo: {df['success_score'].min():.2f}")
    print(f"   📊 Máximo: {df['success_score'].max():.2f}")

    # Mostrar música com maior score
    if len(df) > 0:
        top_score_idx = df['success_score'].idxmax()
        print(f"   🏆 Música com maior score: {df.loc[top_score_idx, 'track_name'][:40]}...")
else:
    print("   ❌ Colunas 'rank' ou 'streams_millions' não encontradas!")

# ===================================================================
# 4. PERFIL MUSICAL (categorização adaptada para Américas)
# ===================================================================
print("\n4️⃣ Perfil musical (categorização para mercado latino):")
print("-" * 40)

def classificar_perfil_americas(row):
    """
    Classifica a música baseado em características musicais
    Adaptado para capturar preferências latino-americanas
    """
    # Música latina típica (alta danceability + energia moderada)
    if row['danceability'] > 0.75 and row['energy'] > 0.65:
        return '🔥 Ritmo Latino (Dançante/Energético)'
    # Reggaeton / Urbano
    elif row['danceability'] > 0.8 and row['valence'] > 0.6:
        return '🎵 Reggaeton/Urbano'
    # Balada romântica (baixa danceability, alta valence)
    elif row['danceability'] < 0.5 and row['valence'] > 0.6:
        return '💕 Balada Romântica'
    # Música triste (baixa valence)
    elif row['valence'] < 0.3:
        return '😢 Melancólica/Triste'
    # Música alegre (alta valence)
    elif row['valence'] > 0.7:
        return '😊 Alegre/Positiva'
    # Alta energia
    elif row['energy'] > 0.7:
        return '⚡ Alta Energia'
    # Música dançante
    elif row['danceability'] > 0.7:
        return '💃 Dançante'
    # Neutro
    else:
        return '🎵 Neutro/Equilibrado'

if all(col in df.columns for col in ['danceability', 'energy', 'valence']):
    df['music_profile'] = df.apply(classificar_perfil_americas, axis=1)
    print(f"   ✅ music_profile criada")
    print(f"\n   📊 Distribuição dos perfis nas Américas:")
    perfis = df['music_profile'].value_counts()
    for perfil, qtd in perfis.items():
        print(f"      {perfil:25} : {qtd:8,} ({qtd/len(df)*100:.1f}%)")
else:
    print("   ❌ Colunas musicais não encontradas!")

# ===================================================================
# 5. ESTAÇÃO DO ANO (adaptada para hemisfério norte/sul)
# ===================================================================
print("\n5️⃣ Estação do ano (adaptado para América do Sul vs Norte):")
print("-" * 40)

def get_season_americas(month, country):
    """
    Determina estação considerando hemisfério do país
    """
    # Países do hemisfério sul
    hemisferio_sul = ['Brazil', 'Brasil', 'Argentina', 'Chile', 'Peru',
                      'Bolivia', 'Paraguay', 'Uruguay', 'Ecuador']

    # Verificar se país está no hemisfério sul
    is_south = country in hemisferio_sul

    if is_south:
        # Hemisfério Sul (estações opostas)
        if month in [12, 1, 2]:
            return '☀️ Verão (HS)'
        elif month in [3, 4, 5]:
            return '🍂 Outono (HS)'
        elif month in [6, 7, 8]:
            return '❄️ Inverno (HS)'
        else:
            return '🌸 Primavera (HS)'
    else:
        # Hemisfério Norte
        if month in [12, 1, 2]:
            return '❄️ Inverno (HN)'
        elif month in [3, 4, 5]:
            return '🌸 Primavera (HN)'
        elif month in [6, 7, 8]:
            return '☀️ Verão (HN)'
        else:
            return '🍂 Outono (HN)'

if 'month' in df.columns and 'country' in df.columns:
    df['season'] = df.apply(lambda row: get_season_americas(row['month'], row['country']), axis=1)
    print(f"   ✅ season criada (com distinção hemisfério)")
    print(f"\n   📊 Distribuição das estações nas Américas:")
    estacoes = df['season'].value_counts()
    for estacao, qtd in estacoes.items():
        print(f"      {estacao:20} : {qtd:8,} ({qtd/len(df)*100:.1f}%)")
else:
    print("   ❌ Colunas 'month' ou 'country' não encontradas!")

# ===================================================================
# 6. GRUPO DE RANK (para análise agregada)
# ===================================================================
print("\n6️⃣ Grupo de rank:")
print("-" * 40)

if 'rank' in df.columns:
    df['rank_group'] = pd.cut(
        df['rank'],
        bins=[0, 10, 50, 100, 200],
        labels=['🏆 Top 10', '📊 Top 11-50', '📈 Top 51-100', '📉 Top 101-200']
    )
    print(f"   ✅ rank_group criada")
    print(f"\n   📊 Distribuição dos grupos nas Américas:")
    grupos = df['rank_group'].value_counts().sort_index()
    for grupo, qtd in grupos.items():
        print(f"      {grupo:20} : {qtd:8,} ({qtd/len(df)*100:.1f}%)")
else:
    print("   ❌ Coluna 'rank' não encontrada!")

# ===================================================================
# 7. SUB-REGIÃO DAS AMÉRICAS (Norte, Central, Sul)
# ===================================================================
print("\n7️⃣ Sub-região das Américas:")
print("-" * 40)

def get_subregion(country):
    """Classifica países por sub-região das Américas"""
    # América do Norte
    if country in ['United States', 'USA', 'Canada', 'Mexico']:
        return '🌎 América do Norte'
    # América Central
    elif country in ['Guatemala', 'Belize', 'Honduras', 'El Salvador', 'Nicaragua',
                      'Costa Rica', 'Panama']:
        return '🌏 América Central'
    # Caribe
    elif country in ['Cuba', 'Jamaica', 'Haiti', 'Dominican Republic', 'Puerto Rico',
                      'Bahamas', 'Trinidad and Tobago', 'Barbados']:
        return '🏝️ Caribe'
    # América do Sul
    elif country in ['Brazil', 'Brasil', 'Argentina', 'Chile', 'Peru', 'Colombia',
                      'Venezuela', 'Ecuador', 'Bolivia', 'Paraguay', 'Uruguay']:
        return '🗺️ América do Sul'
    else:
        return '📍 Outros'

if 'country' in df.columns:
    df['sub_region'] = df['country'].apply(get_subregion)
    print(f"   ✅ sub_region criada")
    print(f"\n   📊 Distribuição por sub-região:")
    regioes = df['sub_region'].value_counts()
    for regiao, qtd in regioes.items():
        print(f"      {regiao:20} : {qtd:8,} ({qtd/len(df)*100:.1f}%)")
else:
    print("   ❌ Coluna 'country' não encontrada!")

# ===================================================================
# 8. FAIXA DE STREAMS (categorização)
# ===================================================================
print("\n8️⃣ Faixa de streams (popularidade):")
print("-" * 40)

if 'streams_millions' in df.columns:
    def get_popularity_category(streams_m):
        if streams_m >= 100:
            return '🌟 Mega Hit (>100M)'
        elif streams_m >= 50:
            return '💎 Super Hit (50-100M)'
        elif streams_m >= 10:
            return '⭐ Hit (10-50M)'
        elif streams_m >= 1:
            return '📈 Sucesso (1-10M)'
        else:
            return '🆕 Novo (<1M)'

    df['popularity_category'] = df['streams_millions'].apply(get_popularity_category)
    print(f"   ✅ popularity_category criada")
    print(f"\n   📊 Distribuição de popularidade:")
    popularidades = df['popularity_category'].value_counts()
    for cat, qtd in popularidades.items():
        print(f"      {cat:25} : {qtd:8,} ({qtd/len(df)*100:.1f}%)")
else:
    print("   ❌ Coluna 'streams_millions' não encontrada!")

# ===================================================================
# 9. INDICADOR DE MÚSICA LATINA (baseado em características)
# ===================================================================
print("\n9️⃣ Indicador de música latina:")
print("-" * 40)

if all(col in df.columns for col in ['danceability', 'energy', 'valence']):
    # Música latina típica: alta danceability, energia média-alta, valence positiva
    df['is_latin_style'] = ((df['danceability'] > 0.7) &
                           (df['energy'] > 0.6) &
                           (df['valence'] > 0.5)).astype(int)

    percentual_latin = df['is_latin_style'].mean() * 100
    print(f"   ✅ is_latin_style criada")
    print(f"   📊 Músicas com estilo latino: {percentual_latin:.1f}% do total")

    # Por país
    latin_by_country = df.groupby('country')['is_latin_style'].mean().sort_values(ascending=False)
    print(f"\n   🎵 Países com maior % de estilo latino:")
    for pais, pct in latin_by_country.head(5).items():
        print(f"      {pais:20} : {pct*100:.1f}%")
else:
    print("   ❌ Colunas musicais não encontradas!")

# ===================================================================
# 10. RESULTADO FINAL
# ===================================================================
print("\n" + "="*60)
print("✅ FEATURE ENGINEERING CONCLUÍDO!")
print("="*60)

print(f"""
📊 RESUMO FINAL:
   • Shape original: {df.shape[0]:,} registros
   • Shape atual: {df.shape[0]:,} registros
   • Colunas originais: {len(COLUNAS_SELECIONADAS) if 'COLUNAS_SELECIONADAS' in dir() else 'N/A'}
   • Colunas atuais: {len(df.columns)}
   • Novas features criadas: {len(df.columns) - (len(COLUNAS_SELECIONADAS) if 'COLUNAS_SELECIONADAS' in dir() else 0)}

🎯 FEATURES CRIADAS:
   • streams_millions - Streams em milhões
   • year, month, month_name - Componentes da data
   • day_of_week, day_name - Dia da semana
   • week_number - Número da semana
   • success_score - Score de sucesso
   • music_profile - Perfil musical categorizado
   • season - Estação do ano (com hemisfério)
   • rank_group - Grupo de rank
   • sub_region - Sub-região das Américas
   • popularity_category - Faixa de popularidade
   • is_latin_style - Indicador de estilo latino
""")

# Mostrar amostra das novas colunas
print("\n📋 AMOSTRA DAS NOVAS FEATURES:")
novas_colunas = ['track_name', 'country', 'sub_region', 'streams_millions',
                 'music_profile', 'popularity_category', 'success_score', 'is_latin_style']
colunas_existentes = [col for col in novas_colunas if col in df.columns]

if colunas_existentes:
    print(df[colunas_existentes].head(10).to_string())
else:
    print("Nenhuma das novas colunas encontrada para exibir.")

🔨 FEATURE ENGINEERING - ANÁLISE DAS AMÉRICAS
📊 Shape inicial: (245338, 12)
🌍 Países presentes: 17

1️⃣ Streams em milhões:
----------------------------------------
   ✅ streams_millions criada
   📊 Total de streams nas Américas: 106547.0M
   📊 Média por música: 0.43M

2️⃣ Componentes da data:
----------------------------------------
   ✅ year: 2021 - 2022
   ✅ month: criada
   ✅ month_name: April
   ✅ day_of_week: 0=Segunda, 6=Domingo
   ✅ day_name: criada
   ✅ week_number: criada

3️⃣ Score de sucesso (Rank ajustado):
----------------------------------------
   Fórmula: (201 - rank) * log(streams_millions)
   Quanto MAIOR o score, MELHOR a música
   ✅ success_score criada
   📊 Média: 33.14
   📊 Mediana: 8.80
   📊 Mínimo: 0.01
   📊 Máximo: 374.97
   🏆 Música com maior score: Fiel...

4️⃣ Perfil musical (categorização para mercado latino):
----------------------------------------
   ✅ music_profile criada

   📊 Distribuição dos perfis nas Américas:
      🔥 Ritmo Latino (Dançante/Energét

In [ ]:
# ===================================================================
# VALIDAÇÃO FINAL DOS DADOS LIMPOS - AMÉRICAS
# ===================================================================
# Explicação: Verificar se a limpeza foi bem-sucedida e gerar métricas
# específicas para o continente americano

print("✅ VALIDAÇÃO FINAL DOS DADOS - AMÉRICAS")
print("=" * 60)

# Verificar se df existe
if df is None or len(df) == 0:
    print("❌ DataFrame vazio! Execute o carregamento dos dados primeiro.")
    raise ValueError("DataFrame não carregado")

# ===================================================================
# 1. RESUMO GERAL DAS AMÉRICAS
# ===================================================================
print("\n1️⃣ RESUMO GERAL DAS AMÉRICAS:")
print("-" * 50)

# Métricas básicas
total_registros = len(df)
total_colunas = len(df.columns)
memoria_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"📊 Shape final: {total_registros:,} linhas × {total_colunas} colunas")
print(f"💾 Memória utilizada: {memoria_mb:.2f} MB")

# Período
if 'week' in df.columns and df['week'].notna().any():
    print(f"📅 Período: {df['week'].min().date()} até {df['week'].max().date()}")
    dias_total = (df['week'].max() - df['week'].min()).days
    print(f"   Duração: {dias_total} dias")
else:
    print(f"📅 Período: Não disponível")

# Contagens únicas
print(f"\n📊 CONTAGENS ÚNICAS:")
print(f"   🌍 Países únicos: {df['country'].nunique()}")
print(f"   🎵 Músicas únicas: {df['track_name'].nunique():,}")
print(f"   🎤 Artistas únicos: {df['artist_names'].nunique():,}")
if 'sub_region' in df.columns:
    print(f"   🗺️ Sub-regiões: {df['sub_region'].nunique()}")

# ===================================================================
# 2. DISTRIBUIÇÃO GEOGRÁFICA (AMÉRICAS)
# ===================================================================
print("\n2️⃣ DISTRIBUIÇÃO GEOGRÁFICA:")
print("-" * 50)

if 'country' in df.columns:
    # Top países das Américas
    top_paises = df.groupby('country').size().sort_values(ascending=False).head(10)
    print("🏆 TOP 10 PAÍSES POR REGISTROS:")
    for i, (pais, qtd) in enumerate(top_paises.items(), 1):
        pct = (qtd / total_registros) * 100
        barra = '█' * int(pct / 2)
        print(f"   {i:2}. {pais:20} : {qtd:6,} registros ({pct:.1f}%) {barra}")

# Sub-regiões
if 'sub_region' in df.columns:
    print("\n🗺️ DISTRIBUIÇÃO POR SUB-REGIÃO:")
    regioes = df['sub_region'].value_counts()
    for regiao, qtd in regioes.items():
        pct = (qtd / total_registros) * 100
        barra = '█' * int(pct / 2)
        print(f"   {regiao:20} : {qtd:6,} registros ({pct:.1f}%) {barra}")

# ===================================================================
# 3. VERIFICAÇÕES DE INTEGRIDADE
# ===================================================================
print("\n3️⃣ VERIFICAÇÕES DE INTEGRIDADE:")
print("-" * 50)

checks = {}

# Verificar nulos
nulos_total = df.isnull().sum().sum()
checks["Sem valores nulos"] = nulos_total == 0
if not checks["Sem valores nulos"]:
    print(f"   ⚠️ ATENÇÃO: {nulos_total} valores nulos encontrados!")
    colunas_com_nulos = df.isnull().sum()
    colunas_com_nulos = colunas_com_nulos[colunas_com_nulos > 0]
    for col, qtd in colunas_com_nulos.items():
        print(f"      • {col}: {qtd} nulos ({qtd/len(df)*100:.2f}%)")

# Rank
if 'rank' in df.columns:
    checks["Rank entre 1 e 200"] = df['rank'].between(1, 200).all()
    if not checks["Rank entre 1 e 200"]:
        print(f"   ⚠️ Rank inválido: min={df['rank'].min()}, max={df['rank'].max()}")

# Streams
if 'streams' in df.columns:
    checks["Streams positivos"] = (df['streams'] > 0).all()
    if not checks["Streams positivos"]:
        print(f"   ⚠️ Streams negativos: {(df['streams'] <= 0).sum()} registros")

# Datas
if 'week' in df.columns:
    checks["Datas válidas"] = df['week'].notna().all()
    if not checks["Datas válidas"]:
        print(f"   ⚠️ Datas inválidas: {df['week'].isna().sum()} registros")

# Características musicais
for col in ['danceability', 'energy', 'valence']:
    if col in df.columns:
        check_name = f"{col.capitalize()} entre 0 e 1"
        checks[check_name] = df[col].between(0, 1).all()
        if not checks[check_name]:
            invalidos_abaixo = (df[col] < 0).sum()
            invalidos_acima = (df[col] > 1).sum()
            print(f"   ⚠️ {col}: {invalidos_abaixo} abaixo de 0, {invalidos_acima} acima de 1")

# Exibir resultados dos checks
print("\n📋 RESULTADO DAS VERIFICAÇÕES:")
for check, result in checks.items():
    status = "✅" if result else "❌"
    print(f"   {status} {check}")

# ===================================================================
# 4. ESTATÍSTICAS DETALHADAS POR PAÍS
# ===================================================================
print("\n4️⃣ ESTATÍSTICAS DETALHADAS POR PAÍS (Top 8):")
print("-" * 70)

if all(col in df.columns for col in ['country', 'streams_millions', 'danceability']):
    # Agrupar por país
    stats_paises = df.groupby('country').agg({
        'streams_millions': ['sum', 'mean', 'count'],
        'danceability': 'mean',
        'energy': 'mean',
        'valence': 'mean'
    }).round(2)

    stats_paises.columns = ['total_streams', 'media_streams', 'registros',
                            'danceability', 'energy', 'valence']
    stats_paises = stats_paises.sort_values('total_streams', ascending=False).head(8)

    print(f"{'País':20} | {'Total Streams':>12} | {'Dança':>5} | {'Energia':>5} | {'Valence':>5}")
    print("-" * 70)
    for pais, row in stats_paises.iterrows():
        print(f"{pais:20} | {row['total_streams']:>8.1f}M   | {row['danceability']:>5.2f} | {row['energy']:>5.2f} | {row['valence']:>5.2f}")

# ===================================================================
# 5. ESTATÍSTICAS GERAIS (Todas as Américas)
# ===================================================================
print("\n5️⃣ ESTATÍSTICAS GERAIS - TODAS AS AMÉRICAS:")
print("-" * 50)

if 'streams_millions' in df.columns:
    total_streams = df['streams_millions'].sum()
    print(f"\n💰 STREAMS:")
    print(f"   💵 Total: {total_streams:,.1f} milhões")
    print(f"   📊 Média por música: {df['streams_millions'].mean():.2f}M")
    print(f"   📈 Mediana: {df['streams_millions'].median():.2f}M")
    print(f"   📉 Desvio padrão: {df['streams_millions'].std():.2f}M")
    print(f"   🎯 Mínimo: {df['streams_millions'].min():.2f}M")
    print(f"   🏆 Máximo: {df['streams_millions'].max():.2f}M")

if all(col in df.columns for col in ['danceability', 'energy', 'valence']):
    print(f"\n🎵 CARACTERÍSTICAS MUSICAIS (0-1):")
    print(f"   💃 Danceability: média={df['danceability'].mean():.3f} ± {df['danceability'].std():.3f}")
    print(f"   ⚡ Energy: média={df['energy'].mean():.3f} ± {df['energy'].std():.3f}")
    print(f"   😊 Valence: média={df['valence'].mean():.3f} ± {df['valence'].std():.3f}")

if 'success_score' in df.columns:
    print(f"\n🏆 SCORE DE SUCESSO:")
    print(f"   Média: {df['success_score'].mean():.2f}")
    print(f"   Mediana: {df['success_score'].median():.2f}")
    print(f"   Mínimo: {df['success_score'].min():.2f}")
    print(f"   Máximo: {df['success_score'].max():.2f}")

if 'is_latin_style' in df.columns:
    pct_latin = df['is_latin_style'].mean() * 100
    print(f"\n🎵 ESTILO LATINO:")
    print(f"   Percentual de músicas com estilo latino: {pct_latin:.1f}%")

# ===================================================================
# 6. AMOSTRA DOS DADOS LIMPOS
# ===================================================================
print("\n6️⃣ AMOSTRA DOS DADOS LIMPOS (8 linhas aleatórias):")
print("-" * 70)

# Selecionar colunas para mostrar
colunas_mostrar = ['track_name', 'artist_names', 'country', 'sub_region' if 'sub_region' in df.columns else 'country',
                   'rank', 'streams_millions', 'danceability', 'energy', 'valence',
                   'music_profile' if 'music_profile' in df.columns else None]

colunas_existentes = [col for col in colunas_mostrar if col is not None and col in df.columns]

if colunas_existentes:
    # Pegar amostra aleatória
    amostra = df[colunas_existentes].sample(min(8, len(df)))
    print(amostra.to_string())
else:
    print("Nenhuma coluna disponível para exibir")

# ===================================================================
# 7. QUALIDADE DOS DADOS POR PAÍS
# ===================================================================
print("\n7️⃣ QUALIDADE DOS DADOS POR PAÍS:")
print("-" * 50)

if 'country' in df.columns:
    # Verificar países com poucos dados
    contagem_paises = df.groupby('country').size()
    paises_poucos_dados = contagem_paises[contagem_paises < 100]

    if len(paises_poucos_dados) > 0:
        print(f"⚠️ Países com menos de 100 registros ({len(paises_poucos_dados)}):")
        for pais, qtd in paises_poucos_dados.head(10).items():
            print(f"   • {pais}: {qtd} registros")
    else:
        print("✅ Todos os países têm mais de 100 registros")

    # Verificar países sem dados de características musicais
    if 'danceability' in df.columns:
        paises_sem_caracteristicas = df[df['danceability'].isna()]['country'].unique()
        if len(paises_sem_caracteristicas) > 0:
            print(f"\n⚠️ Países com dados musicais faltando: {list(paises_sem_caracteristicas)}")

# ===================================================================
# 8. COMPARAÇÃO AMÉRICA DO NORTE VS SUL
# ===================================================================
print("\n8️⃣ COMPARAÇÃO AMÉRICA DO NORTE VS AMÉRICA DO SUL:")
print("-" * 50)

if 'sub_region' in df.columns and 'streams_millions' in df.columns:
    america_norte = df[df['sub_region'] == '🌎 América do Norte']
    america_sul = df[df['sub_region'] == '🗺️ América do Sul']

    if len(america_norte) > 0 and len(america_sul) > 0:
        print(f"\n📊 AMÉRICA DO NORTE ({len(america_norte):,} registros):")
        print(f"   Média streams: {america_norte['streams_millions'].mean():.2f}M")
        print(f"   Danceability: {america_norte['danceability'].mean():.3f}")
        print(f"   Energy: {america_norte['energy'].mean():.3f}")
        print(f"   Valence: {america_norte['valence'].mean():.3f}")

        print(f"\n📊 AMÉRICA DO SUL ({len(america_sul):,} registros):")
        print(f"   Média streams: {america_sul['streams_millions'].mean():.2f}M")
        print(f"   Danceability: {america_sul['danceability'].mean():.3f}")
        print(f"   Energy: {america_sul['energy'].mean():.3f}")
        print(f"   Valence: {america_sul['valence'].mean():.3f}")

        # Diferenças
        print(f"\n📈 DIFERENÇAS (Norte vs Sul):")
        print(f"   Danceability: {'Norte > Sul' if america_norte['danceability'].mean() > america_sul['danceability'].mean() else 'Sul > Norte'}")
        print(f"   Energia: {'Norte > Sul' if america_norte['energy'].mean() > america_sul['energy'].mean() else 'Sul > Norte'}")
        print(f"   Positividade: {'Norte > Sul' if america_norte['valence'].mean() > america_sul['valence'].mean() else 'Sul > Norte'}")
    else:
        print("⚠️ Dados insuficientes para comparação")

# ===================================================================
# 9. MÉTRICAS DE DESEMPENHO DA LIMPEZA
# ===================================================================
print("\n9️⃣ MÉTRICAS DE DESEMPENHO DA LIMPEZA:")
print("-" * 50)

if 'df_raw' in dir():
    tamanho_original = len(df_raw) if 'df_raw' in locals() else 0
    tamanho_final = len(df)
    reducao = ((tamanho_original - tamanho_final) / tamanho_original) * 100 if tamanho_original > 0 else 0

    print(f"📊 Registros originais: {tamanho_original:,}")
    print(f"📊 Registros finais (Américas): {tamanho_final:,}")
    print(f"📉 Redução: {reducao:.1f}% (dados de outros continentes removidos)")
else:
    print("⚠️ Não foi possível calcular métricas de redução (df_raw não disponível)")

# ===================================================================
# 10. CONCLUSÃO DA VALIDAÇÃO
# ===================================================================
print("\n" + "="*60)
print("🎯 CONCLUSÃO DA VALIDAÇÃO")
print("="*60)

# Verificar se todos os checks passaram
todos_checks_passaram = all(v == True for v in checks.values() if isinstance(v, bool))

if todos_checks_passaram:
    print("""
✅ VALIDAÇÃO CONCLUÍDA COM SUCESSO!
   • Todos os testes de integridade passaram
   • Dados das Américas estão limpos e consistentes
   • Dataset pronto para análise e visualizações
""")
else:
    print("""
⚠️ VALIDAÇÃO COM RESSALVAS!
   • Alguns testes de integridade falharam
   • Verifique os alertas acima
   • Pode ser necessário ajustes adicionais
""")

# Estatísticas finais resumidas
print("📊 RESUMO FINAL DAS AMÉRICAS:")
print("-" * 50)
print(f"   🌎 Países: {df['country'].nunique()}")
print(f"   🎵 Músicas: {df['track_name'].nunique():,}")
print(f"   🎤 Artistas: {df['artist_names'].nunique():,}")
print(f"   💰 Total streams: {df['streams_millions'].sum():.1f}M" if 'streams_millions' in df.columns else "   💰 Total streams: N/A")
print(f"   💃 Dançabilidade média: {df['danceability'].mean():.3f}" if 'danceability' in df.columns else "   💃 Dançabilidade: N/A")
print(f"   📅 Período: {df['week'].min().date()} a {df['week'].max().date()}" if 'week' in df.columns else "   📅 Período: N/A")

print("\n✅ VALIDAÇÃO CONCLUÍDA - DADOS DAS AMÉRICAS PRONTOS PARA ANÁLISE!")
print("="*60)

✅ VALIDAÇÃO FINAL DOS DADOS - AMÉRICAS

1️⃣ RESUMO GERAL DAS AMÉRICAS:
--------------------------------------------------
📊 Shape final: 245,338 linhas × 26 colunas
💾 Memória utilizada: 214.34 MB
📅 Período: 2021-02-04 até 2022-07-14
   Duração: 525 dias

📊 CONTAGENS ÚNICAS:
   🌍 Países únicos: 17
   🎵 Músicas únicas: 5,479
   🎤 Artistas únicos: 3,481
   🗺️ Sub-regiões: 4

2️⃣ DISTRIBUIÇÃO GEOGRÁFICA:
--------------------------------------------------
🏆 TOP 10 PAÍSES POR REGISTROS:
    1. Canada               : 15,199 registros (6.2%) ███
    2. Uruguay              : 15,198 registros (6.2%) ███
    3. Bolivia              : 15,198 registros (6.2%) ███
    4. Costa Rica           : 15,198 registros (6.2%) ███
    5. Colombia             : 15,198 registros (6.2%) ███
    6. Dominican Republic   : 15,198 registros (6.2%) ███
    7. Ecuador              : 15,198 registros (6.2%) ███
    8. Panama               : 15,198 registros (6.2%) ███
    9. Guatemala            : 15,198 registros (6.

In [ ]:
# ===================================================================
# SALVAR RESULTADOS - DADOS DAS AMÉRICAS
# ===================================================================
# Explicação: Salvar os dados limpos e análises específicas das Américas

print("💾 SALVANDO RESULTADOS - AMÉRICAS")
print("=" * 60)

# Criar pasta para resultados das Américas (opcional)
import os
resultados_dir = '/content/drive/MyDrive/spotify_americas_results/'
try:
    os.makedirs(resultados_dir, exist_ok=True)
    print(f"📁 Pasta criada: {resultados_dir}")
except:
    print(f"⚠️ Não foi possível criar pasta, salvando na raiz")

# ===================================================================
# 1. SALVAR DADOS LIMPOS (FORMATOS MÚLTIPLOS)
# ===================================================================
print("\n1️⃣ Salvando dados limpos das Américas:")
print("-" * 40)

# Salvar em Parquet (mais eficiente, recomendado)
try:
    df.to_parquet(f'{resultados_dir}spotify_americas_clean.parquet', compression='snappy')
    print(f"   ✅ spotify_americas_clean.parquet salvo")
except Exception as e:
    print(f"   ⚠️ Erro ao salvar Parquet: {e}")

# Salvar backup em CSV (compatibilidade)
try:
    df.to_csv(f'{resultados_dir}spotify_americas_clean.csv', index=False)
    print(f"   ✅ spotify_americas_clean.csv salvo")
except Exception as e:
    print(f"   ⚠️ Erro ao salvar CSV: {e}")

# ===================================================================
# 2. SALVAR DADOS AGREGDOS POR PAÍS
# ===================================================================
print("\n2️⃣ Salvando dados agregados por país:")
print("-" * 40)

try:
    if 'country' in df.columns:
        # Agregação completa por país
        analise_paises = df.groupby('country').agg({
            'streams_millions': ['sum', 'mean', 'std'],
            'danceability': 'mean',
            'energy': 'mean',
            'valence': 'mean',
            'track_name': 'count'
        }).round(3)

        # Renomear colunas para melhor legibilidade
        analise_paises.columns = ['total_streams_m', 'media_streams_m', 'std_streams_m',
                                  'danceability', 'energy', 'valence', 'total_musicas']
        analise_paises = analise_paises.sort_values('total_streams_m', ascending=False)

        # Salvar
        analise_paises.to_csv(f'{resultados_dir}analise_por_pais_americas.csv')
        print(f"   ✅ analise_por_pais_americas.csv salvo ({len(analise_paises)} países)")

        # Mostrar top 5
        print(f"\n   📊 Top 5 países por streams:")
        for pais, row in analise_paises.head(5).iterrows():
            print(f"      {pais:20} : {row['total_streams_m']:.1f}M streams")
except Exception as e:
    print(f"   ⚠️ Erro ao salvar análise por país: {e}")

# ===================================================================
# 3. SALVAR TOP MÚSICAS E TOP ARTISTAS
# ===================================================================
print("\n3️⃣ Salvando top músicas e artistas:")
print("-" * 40)

try:
    # Top 100 músicas das Américas
    top_musicas = (df.groupby(['track_name', 'artist_names'])
                   .agg({
                       'streams_millions': 'sum',
                       'rank': 'min',
                       'country': 'nunique'
                   })
                   .rename(columns={'country': 'paises_alcancados'})
                   .sort_values('streams_millions', ascending=False)
                   .head(100)
                   .reset_index())

    top_musicas.to_csv(f'{resultados_dir}top_100_musicas_americas.csv', index=False)
    print(f"   ✅ top_100_musicas_americas.csv salvo")

    # Top 50 artistas das Américas
    top_artistas = (df.groupby('artist_names')
                    .agg({
                        'streams_millions': 'sum',
                        'track_name': 'nunique',
                        'country': 'nunique'
                    })
                    .rename(columns={'track_name': 'musicas_distintas', 'country': 'paises_alcancados'})
                    .sort_values('streams_millions', ascending=False)
                    .head(50)
                    .reset_index())

    top_artistas.to_csv(f'{resultados_dir}top_50_artistas_americas.csv', index=False)
    print(f"   ✅ top_50_artistas_americas.csv salvo")

except Exception as e:
    print(f"   ⚠️ Erro ao salvar top músicas/artistas: {e}")

# ===================================================================
# 4. SALVAR ESTATÍSTICAS POR SUB-REGIÃO
# ===================================================================
print("\n4️⃣ Salvando estatísticas por sub-região:")
print("-" * 40)

if 'sub_region' in df.columns:
    try:
        stats_regioes = df.groupby('sub_region').agg({
            'streams_millions': ['sum', 'mean', 'count'],
            'danceability': 'mean',
            'energy': 'mean',
            'valence': 'mean',
            'is_latin_style': 'mean' if 'is_latin_style' in df.columns else 'count'
        }).round(3)

        stats_regioes.to_csv(f'{resultados_dir}estatisticas_por_regiao_americas.csv')
        print(f"   ✅ estatisticas_por_regiao_americas.csv salvo")
    except Exception as e:
        print(f"   ⚠️ Erro ao salvar estatísticas por região: {e}")

# ===================================================================
# 5. SALVAR PERFIL MUSICAL POR PAÍS
# ===================================================================
print("\n5️⃣ Salvando perfil musical por país:")
print("-" * 40)

if 'music_profile' in df.columns:
    try:
        # Matriz de perfil musical por país
        perfil_paises = pd.crosstab(df['country'], df['music_profile'], normalize='index') * 100
        perfil_paises.round(2).to_csv(f'{resultados_dir}perfil_musical_por_pais_americas.csv')
        print(f"   ✅ perfil_musical_por_pais_americas.csv salvo")
    except Exception as e:
        print(f"   ⚠️ Erro ao salvar perfil musical: {e}")

# ===================================================================
# 6. SALVAR METADADOS COMPLETOS
# ===================================================================
print("\n6️⃣ Salvando metadados do projeto:")
print("-" * 40)

try:
    # Metadados básicos
    metadados = {
        'projeto': 'Spotify Weekly Top 200 - Análise das Américas',
        'data_processamento': pd.Timestamp.now().isoformat(),
        'shape': list(df.shape),
        'colunas': list(df.columns),
        'memoria_mb': float(df.memory_usage(deep=True).sum() / 1024**2),
        'periodo_inicio': df['week'].min().isoformat() if 'week' in df.columns else None,
        'periodo_fim': df['week'].max().isoformat() if 'week' in df.columns else None,
        'total_streams_millions': float(df['streams_millions'].sum()) if 'streams_millions' in df.columns else None,
        'paises_unicos': int(df['country'].nunique()) if 'country' in df.columns else None,
        'musicas_unicas': int(df['track_name'].nunique()) if 'track_name' in df.columns else None,
        'artistas_unicos': int(df['artist_names'].nunique()) if 'artist_names' in df.columns else None,
        'media_danceability': float(df['danceability'].mean()) if 'danceability' in df.columns else None,
        'media_energy': float(df['energy'].mean()) if 'energy' in df.columns else None,
        'media_valence': float(df['valence'].mean()) if 'valence' in df.columns else None,
        'paises_lista': list(df['country'].unique()) if 'country' in df.columns else None,
        'sub_regioes': list(df['sub_region'].unique()) if 'sub_region' in df.columns else None
    }

    import json
    with open(f'{resultados_dir}spotify_americas_metadados.json', 'w') as f:
        json.dump(metadados, f, indent=2, default=str)
    print(f"   ✅ spotify_americas_metadados.json salvo")

except Exception as e:
    print(f"   ⚠️ Erro ao salvar metadados: {e}")

# ===================================================================
# 7. SALVAR RESUMO EXECUTIVO (formato legível)
# ===================================================================
print("\n7️⃣ Salvando resumo executivo:")
print("-" * 40)

try:
    with open(f'{resultados_dir}resumo_executivo_americas.txt', 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("RESUMO EXECUTIVO - ANÁLISE SPOTIFY AMÉRICAS\n")
        f.write("="*60 + "\n\n")

        f.write(f"📊 DADOS GERAIS:\n")
        f.write(f"   • Total de registros: {len(df):,}\n")
        f.write(f"   • Países analisados: {df['country'].nunique()}\n")
        f.write(f"   • Músicas únicas: {df['track_name'].nunique():,}\n")
        f.write(f"   • Artistas únicos: {df['artist_names'].nunique():,}\n")
        f.write(f"   • Período: {df['week'].min().date()} a {df['week'].max().date()}\n\n")

        if 'streams_millions' in df.columns:
            f.write(f"💰 STREAMS:\n")
            f.write(f"   • Total: {df['streams_millions'].sum():.1f} milhões\n")
            f.write(f"   • Média por música: {df['streams_millions'].mean():.2f}M\n\n")

        if all(col in df.columns for col in ['danceability', 'energy', 'valence']):
            f.write(f"🎵 CARACTERÍSTICAS MUSICAIS:\n")
            f.write(f"   • Dançabilidade: {df['danceability'].mean():.3f}\n")
            f.write(f"   • Energia: {df['energy'].mean():.3f}\n")
            f.write(f"   • Positividade: {df['valence'].mean():.3f}\n\n")

        if 'music_profile' in df.columns:
            f.write(f"🎭 PERFIL MUSICAL DOMINANTE:\n")
            top_perfil = df['music_profile'].value_counts().index[0]
            pct_perfil = df['music_profile'].value_counts().values[0] / len(df) * 100
            f.write(f"   • {top_perfil}: {pct_perfil:.1f}%\n\n")

        if 'sub_region' in df.columns:
            f.write(f"🗺️ DISTRIBUIÇÃO POR SUB-REGIÃO:\n")
            for regiao, qtd in df['sub_region'].value_counts().items():
                pct = qtd / len(df) * 100
                f.write(f"   • {regiao}: {pct:.1f}%\n")

    print(f"   ✅ resumo_executivo_americas.txt salvo")

except Exception as e:
    print(f"   ⚠️ Erro ao salvar resumo executivo: {e}")

# ===================================================================
# 8. SALVAR RELATÓRIO DE QUALIDADE DOS DADOS
# ===================================================================
print("\n8️⃣ Salvando relatório de qualidade dos dados:")
print("-" * 40)

try:
    with open(f'{resultados_dir}relatorio_qualidade_americas.txt', 'w', encoding='utf-8') as f:
        f.write("RELATÓRIO DE QUALIDADE DOS DADOS - AMÉRICAS\n")
        f.write("="*50 + "\n\n")

        # Valores nulos
        nulos = df.isnull().sum()
        nulos = nulos[nulos > 0]
        if len(nulos) > 0:
            f.write("⚠️ VALORES NULOS ENCONTRADOS:\n")
            for col, qtd in nulos.items():
                f.write(f"   • {col}: {qtd} ({qtd/len(df)*100:.2f}%)\n")
        else:
            f.write("✅ Nenhum valor nulo encontrado!\n")

        f.write("\n")

        # Outliers
        if 'streams_millions' in df.columns:
            q1 = df['streams_millions'].quantile(0.25)
            q3 = df['streams_millions'].quantile(0.75)
            iqr = q3 - q1
            outliers = df[(df['streams_millions'] < q1 - 1.5*iqr) | (df['streams_millions'] > q3 + 1.5*iqr)]
            f.write(f"📊 OUTLIERS EM STREAMS:\n")
            f.write(f"   • Q1: {q1:.2f}M\n")
            f.write(f"   • Q3: {q3:.2f}M\n")
            f.write(f"   • Outliers detectados: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)\n")

    print(f"   ✅ relatorio_qualidade_americas.txt salvo")

except Exception as e:
    print(f"   ⚠️ Erro ao salvar relatório de qualidade: {e}")

# ===================================================================
# 9. EXIBIR RESUMO DOS ARQUIVOS SALVOS
# ===================================================================
print("\n" + "="*60)
print("📁 ARQUIVOS SALVOS COM SUCESSO!")
print("="*60)

# Listar arquivos salvos
import os
try:
    arquivos = os.listdir(resultados_dir)
    print(f"\n📂 Pasta: {resultados_dir}")
    print("-" * 40)
    for arquivo in sorted(arquivos):
        tamanho = os.path.getsize(f"{resultados_dir}{arquivo}")
        if tamanho < 1024:
            tamanho_str = f"{tamanho} B"
        elif tamanho < 1024**2:
            tamanho_str = f"{tamanho/1024:.1f} KB"
        else:
            tamanho_str = f"{tamanho/1024**2:.1f} MB"
        print(f"   📄 {arquivo:45} : {tamanho_str}")
except Exception as e:
    print(f"   ⚠️ Erro ao listar arquivos: {e}")

# ===================================================================
# 10. RESUMO FINAL
# ===================================================================
print("\n" + "="*60)
print("✨ SALVAMENTO CONCLUÍDO COM SUCESSO!")
print("="*60)

print(f"""
📊 RESUMO DO QUE FOI SALVO:

   ✅ DADOS PRINCIPAIS:
      • spotify_americas_clean.parquet - Dados limpos (formato eficiente)
      • spotify_americas_clean.csv - Dados limpos (backup)

   ✅ ANÁLISES AGREGADAS:
      • analise_por_pais_americas.csv - Métricas por país
      • top_100_musicas_americas.csv - Top músicas
      • top_50_artistas_americas.csv - Top artistas
      • estatisticas_por_regiao_americas.csv - Análise por sub-região
      • perfil_musical_por_pais_americas.csv - Perfis musicais

   ✅ DOCUMENTAÇÃO:
      • spotify_americas_metadados.json - Metadados do projeto
      • resumo_executivo_americas.txt - Resumo para apresentação
      • relatorio_qualidade_americas.txt - Qualidade dos dados

📂 LOCALIZAÇÃO: {resultados_dir}
""")

💾 SALVANDO RESULTADOS - AMÉRICAS
📁 Pasta criada: /content/drive/MyDrive/spotify_americas_results/

1️⃣ Salvando dados limpos das Américas:
----------------------------------------
   ✅ spotify_americas_clean.parquet salvo
   ✅ spotify_americas_clean.csv salvo

2️⃣ Salvando dados agregados por país:
----------------------------------------
   ✅ analise_por_pais_americas.csv salvo (17 países)

   📊 Top 5 países por streams:
      United States        : 32721.8M streams
      Brazil               : 22017.1M streams
      Mexico               : 18894.4M streams
      Argentina            : 7987.7M streams
      Chile                : 6787.4M streams

3️⃣ Salvando top músicas e artistas:
----------------------------------------
   ✅ top_100_musicas_americas.csv salvo
   ✅ top_50_artistas_americas.csv salvo

4️⃣ Salvando estatísticas por sub-região:
----------------------------------------
   ✅ estatisticas_por_regiao_americas.csv salvo

5️⃣ Salvando perfil musical por país:
----------------